# Final Model

In [ ]:
import pandas as pd
import numpy as np

df_train= pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")
df_test = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv")

X_train = df_train.drop(columns=['purchaseValue'])
y_train = df_train['purchaseValue']

X_test = df_test

def summarize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyzes a DataFrame and returns a summary DataFrame with column information.

    Args:
        df (pd.DataFrame): The input DataFrame to analyze.

    Returns:
        pd.DataFrame: A new DataFrame with the following columns:
                      - "Column": Name of the column from the input DataFrame.
                      - "Type": "Numeric" or "Categoric" based on column dtype.
                      - "NaN_pcnt": Percentage of NaN values in the column.
                      - "N_Unique": Number of unique values in the column.
    """
    summary_data = []

    for col_name in df.columns:
        col_series = df[col_name]

        # Determine Type
        if pd.api.types.is_numeric_dtype(col_series):
            col_type = "Numeric"
        elif pd.api.types.is_string_dtype(col_series) or pd.api.types.is_object_dtype(col_series) or pd.api.types.is_categorical_dtype(col_series):
            col_type = "Categoric"
        else:
            # Handle other types like datetime, boolean, etc.
            # For simplicity, we'll categorize them as "Categoric" if not strictly numeric
            col_type = "Categoric" 

        # Calculate NaN Percentage
        nan_count = col_series.isnull().sum()
        total_rows = len(col_series)
        nan_pcnt = (nan_count / total_rows) * 100 if total_rows > 0 else 0

        # Calculate Number of Unique Values
        n_unique = col_series.nunique(dropna=True) # count NaN as a unique value if present

        summary_data.append({
            "Column": col_name,
            "Type": col_type,
            "NaN_pcnt": nan_pcnt,
            "N_Unique": n_unique
        })

    summary_df = pd.DataFrame(summary_data)
    return summary_df

print(f"Stats summary for X_train:\n{summarize_dataframe_columns(X_train)}")
print("\n" + "="*50 + "\n")
print(f"Stats summary for X_test:\n{summarize_dataframe_columns(X_test)}")


In [ ]:
print("Original X_train shape:", X_train.shape)
print("Original X_test shape:", X_test.shape)

# --- Step 1: Drop columns from X_train and X_test ---
summary_train = summarize_dataframe_columns(X_train)

cols_to_drop = []

# Condition 1: nan_pcnt > 90
cols_to_drop.extend(summary_train[summary_train['NaN_pcnt'] > 90]['Column'].tolist())

# Condition 2: nan_pcnt = 0 and n_unique = 1
cols_to_drop.extend(summary_train[(summary_train['NaN_pcnt'] == 0) & (summary_train['N_Unique'] == 1)]['Column'].tolist())

# Condition 3: n_unique > 100000
cols_to_drop.extend(summary_train[summary_train['N_Unique'] > 100000]['Column'].tolist())

# Remove duplicates from the list of columns to drop
cols_to_drop = list(set(cols_to_drop))

print(f"\nColumns identified for dropping from X_train (and X_test): {cols_to_drop}")

X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

print("X_train shape after Step 1:", X_train.shape)
print("X_test shape after Step 1:", X_test.shape)

# Re-summarize X_train after dropping columns for subsequent steps
summary_train_step1 = summarize_dataframe_columns(X_train)

# --- Step 2: Handle unknown categories in X_test ---
print("\n--- Step 2: Handling unknown categories in X_test ---")
categorical_cols_train_after_drop = summary_train_step1[summary_train_step1['Type'] == 'Categoric']['Column'].tolist()

for col in categorical_cols_train_after_drop:
    if col in X_test.columns: # Ensure the column exists in X_test
        # Get unique categories from training data (excluding NaN)
        train_categories = set(X_train[col].dropna().unique())

        # Identify unknown categories in test data
        # Use .copy() to avoid SettingWithCopyWarning
        # The loc assignment might still trigger the warning if X_test is a slice, but it's generally safe here.
        unknown_mask = ~X_test[col].isin(train_categories) & X_test[col].notna()
        if unknown_mask.any():
            print(f"  Column '{col}': Found unknown categories in X_test. Replacing with NaN.")
            # Replace unknown categories with NaN
            X_test.loc[unknown_mask, col] = np.nan
        else:
            print(f"  Column '{col}': No unknown categories found in X_test.")

print("X_test shape after Step 2:", X_test.shape)


# --- Step 3: Specific Imputation/Encoding for n_unique=1 and nan_pcnt!=0 ---
print("\n--- Step 3: Specific Imputation/Encoding ---")
# Re-summarize X_train to get updated NaN percentages after potential dropping/processing
summary_train_step2 = summarize_dataframe_columns(X_train)

# Identify columns for this specific transformation from X_train
numeric_one_unique_with_nan = summary_train_step2[
    (summary_train_step2['Type'] == 'Numeric') &
    (summary_train_step2['N_Unique'] == 1) &
    (summary_train_step2['NaN_pcnt'] != 0)
]['Column'].tolist()

categoric_one_unique_with_nan = summary_train_step2[
    (summary_train_step2['Type'] == 'Categoric') &
    (summary_train_step2['N_Unique'] == 1) &
    (summary_train_step2['NaN_pcnt'] != 0)
]['Column'].tolist()

# Store unique categorical values for consistent encoding
cat_one_unique_val_map = {}
for col in categoric_one_unique_with_nan:
    # Get the single non-NaN unique value from X_train before transformation
    if not X_train[col].dropna().empty:
        cat_one_unique_val_map[col] = X_train[col].dropna().unique()[0]
    else:
        print(f"  Warning: Categorical column '{col}' has no non-NaN unique value in X_train. Skipping encoding for it.")
        # Remove from list if no unique value found
        categoric_one_unique_with_nan.remove(col)


print(f"Numeric columns (n_unique=1, nan_pcnt!=0) to impute with 0: {numeric_one_unique_with_nan}")
print(f"Categoric columns (n_unique=1, nan_pcnt!=0) to encode: {categoric_one_unique_with_nan}")


# Apply transformations to X_train
for col in numeric_one_unique_with_nan:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna(0)
        print(f"  X_train: Numeric column '{col}' imputed with 0.")

for col in categoric_one_unique_with_nan:
    if col in X_train.columns and col in cat_one_unique_val_map:
        unique_val = cat_one_unique_val_map[col]
        X_train[col] = X_train[col].apply(lambda x: 1 if x == unique_val else 0)
        print(f"  X_train: Categoric column '{col}' encoded (1 for '{unique_val}', 0 for NaN/other).")
    elif col in X_train.columns: # Case where it was in list but no unique val found
        print(f"  X_train: Skipping encoding for '{col}' due to no unique non-NaN value.")


# Apply transformations to X_test (without checking conditions)
for col in numeric_one_unique_with_nan:
    if col in X_test.columns: # Ensure the column exists in X_test
        X_test[col] = X_test[col].fillna(0)
        print(f"  X_test: Numeric column '{col}' imputed with 0 (based on train).")

for col in categoric_one_unique_with_nan:
    if col in X_test.columns and col in cat_one_unique_val_map:
        unique_val = cat_one_unique_val_map[col]
        X_test[col] = X_test[col].apply(lambda x: 1 if x == unique_val else 0)
        print(f"  X_test: Categoric column '{col}' encoded (1 for '{unique_val}', 0 for NaN/other) (based on train).")
    elif col in X_test.columns:
        print(f"  X_test: Skipping encoding for '{col}' due to no unique non-NaN value from train.")


print("\nX_train shape after Step 3:", X_train.shape)
print("X_test shape after Step 3:", X_test.shape)

# --- Step 4: Print columns still containing NaN ---
print("\n--- Step 4: Columns still containing NaN ---")

# Columns eligible for imputation in X_train
nan_cols_train = X_train.columns[X_train.isnull().any()].tolist()
print("Columns eligible for imputation (X_train):", nan_cols_train)

# Columns eligible for imputation in X_test
nan_cols_test = X_test.columns[X_test.isnull().any()].tolist()
print("Columns eligible for imputation (X_test):", nan_cols_test)

In [ ]:
from sklearn.impute import SimpleImputer

# Calculate percentage of NaN values in X_test columns
summary = summarize_dataframe_columns(X_test)
filtered_df = summary[summary["NaN_pcnt"] < 1]

# Identify columns with NaN percentage < 1
cols_to_impute = filtered_df["Column"].tolist()

# Separate numeric and categorical columns
numeric_cols = X_train[cols_to_impute].select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train[cols_to_impute].select_dtypes(exclude=np.number).columns.tolist()

# Initialize imputers
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Fit and transform numeric columns in X_train
if numeric_cols:
    X_train[numeric_cols] = numeric_imputer.fit_transform(X_train[numeric_cols])
    # Transform numeric columns in X_test
    X_test[numeric_cols] = numeric_imputer.transform(X_test[numeric_cols])

# Fit and transform categorical columns in X_train
if categorical_cols:
    X_train[categorical_cols] = categorical_imputer.fit_transform(X_train[categorical_cols])
    # Transform categorical columns in X_test
    X_test[categorical_cols] = categorical_imputer.transform(X_test[categorical_cols])

In [ ]:
X_train['date'] = pd.to_datetime(X_train['date'], format='%Y%m%d')
X_test['date'] = pd.to_datetime(X_test['date'], format='%Y%m%d')

X_train['year'] = X_train['date'].dt.year
X_train['month'] = X_train['date'].dt.month
X_train['day'] = X_train['date'].dt.day
X_train['day_of_week'] = X_train['date'].dt.dayofweek # Monday=0, Sunday=6
X_train['day_of_year'] = X_train['date'].dt.dayofyear
X_train['week_of_year'] = X_train['date'].dt.isocalendar().week.astype(int)
X_train['quarter'] = X_train['date'].dt.quarter
X_train['is_month_start'] = X_train['date'].dt.is_month_start.astype(int)
X_train['is_month_end'] = X_train['date'].dt.is_month_end.astype(int)
X_train['is_weekend'] = X_train['date'].dt.dayofweek.isin([5, 6]).astype(int)

X_test['year'] = X_test['date'].dt.year
X_test['month'] = X_test['date'].dt.month
X_test['day'] = X_test['date'].dt.day
X_test['day_of_week'] = X_test['date'].dt.dayofweek # Monday=0, Sunday=6
X_test['day_of_year'] = X_test['date'].dt.dayofyear
X_test['week_of_year'] = X_test['date'].dt.isocalendar().week.astype(int)
X_test['quarter'] = X_test['date'].dt.quarter
X_test['is_month_start'] = X_test['date'].dt.is_month_start.astype(int)
X_test['is_month_end'] = X_test['date'].dt.is_month_end.astype(int)
X_test['is_weekend'] = X_test['date'].dt.dayofweek.isin([5, 6]).astype(int)

X_train = X_train.drop(['date'],axis=1)
X_test = X_test.drop(['date'],axis=1)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
import joblib

# --- 2. Define Columns to Process ---
# Columns to Label Encode and then KNN Impute
categorical_cols_for_knn = ['trafficSource.keyword', 'trafficSource.referralPath']

# --- 3. Label Encode (leaving NaNs undisturbed) ---

# Dictionary to store fitted LabelEncoders for each column
label_encoders = {}

for col in categorical_cols_for_knn:
    le = LabelEncoder()

    unique_non_null_train = X_train[col].dropna().unique()
    le.fit(unique_non_null_train)
    label_encoders[col] = le

    X_train[col] = X_train[col].apply(lambda x: le.transform([x])[0] if pd.notna(x) else np.nan)

    mapping_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test[col] = X_test[col].map(mapping_dict)


print("\n--- After Label Encoding (NaNs still present) ---")
print("\nNaNs in X_train after Label Encoding:\n", X_train.isnull().sum())
print("\nNaNs in X_test after Label Encoding:\n", X_test.isnull().sum())
print("\nDtypes after Label Encoding:\n", X_train[categorical_cols_for_knn].dtypes)


# --- 4. KNN Impute ---

# Initialize KNNImputer. We will only impute the specific categorical columns after encoding them.
imputer = joblib.load("/kaggle/input/knn-imputer/knn_imputer.joblib")

numeric_cols_for_knn = X_train.select_dtypes(include=np.number).columns.tolist()

# Ensure the columns we want to impute are in this list
for col in categorical_cols_for_knn:
    if col not in numeric_cols_for_knn:
        numeric_cols_for_knn.append(col)

# Sort for consistency if needed, though not strictly required by KNNImputer
numeric_cols_for_knn.sort()


# Fit the imputer on X_train (only the relevant numeric columns)
X_train_imputed_array = imputer.transform(X_train[numeric_cols_for_knn])
X_train_imputed = pd.DataFrame(X_train_imputed_array, columns=numeric_cols_for_knn, index=X_train.index)

# Transform X_test
X_test_imputed_array = imputer.transform(X_test[numeric_cols_for_knn])
X_test_imputed = pd.DataFrame(X_test_imputed_array, columns=numeric_cols_for_knn, index=X_test.index)


# Update the original DataFrames with the imputed values for the target columns
for col in categorical_cols_for_knn:
    X_train[col] = X_train_imputed[col].round().astype(int) # Round and convert to int for imputed labels
    X_test[col] = X_test_imputed[col].round().astype(int) # Round and convert to int for imputed labels


print("\n--- After KNN Imputation (NaNs should be gone in target columns) ---")
print("\nNaNs in X_train after KNN Imputation:\n", X_train.isnull().sum())
print("\nNaNs in X_test after KNN Imputation:\n", X_test.isnull().sum())
print("\nDtypes after KNN Imputation:\n", X_train[categorical_cols_for_knn].dtypes)


# --- 5. Reconvert to Categorical (Object Dtype) ---
for col in categorical_cols_for_knn:
    le = label_encoders[col]

    # Convert the imputed integer labels back to original strings
    X_train[col] = le.inverse_transform(X_train[col])
    X_test[col] = le.inverse_transform(X_test[col])

print("\n--- After Reverting to Categorical Dtype ---")
print("\nFinal NaNs in X_train:\n", X_train.isnull().sum())
print("\nFinal NaNs in X_test:\n", X_test.isnull().sum())
print("\nFinal Dtypes:\n", X_train[categorical_cols_for_knn].dtypes)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler


X_test = X_test[X_train.columns] # Reorder X_test to match X_train's column order

print("Original X_train head:\n", X_train.head())
print("\nOriginal X_test head:\n", X_test.head())
print("\nOriginal X_train dtypes:\n", X_train.dtypes)

# 1. Identify numeric columns
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

# 2. Initialize StandardScaler
scaler = StandardScaler()

# 3. Fit scaler on numeric columns of X_train and transform both X_train and X_test
if numeric_cols: # Only proceed if there are numeric columns
    # Fit and transform X_train numeric columns
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

    # Transform X_test numeric columns
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
    
# Identify categorical columns
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Initialize OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform categorical columns in X_train
if categorical_cols:
    # Fit encoder and transform X_train
    X_train_encoded = pd.DataFrame(
        ohe.fit_transform(X_train[categorical_cols]),
        columns=ohe.get_feature_names_out(categorical_cols),
        index=X_train.index
    )
    
    # Transform categorical columns in X_test
    X_test_encoded = pd.DataFrame(
        ohe.transform(X_test[categorical_cols]),
        columns=ohe.get_feature_names_out(categorical_cols),
        index=X_test.index
    )
    
    # Drop original categorical columns and concatenate encoded ones
    X_train = X_train.drop(columns=categorical_cols).join(X_train_encoded)
    X_test = X_test.drop(columns=categorical_cols).join(X_test_encoded)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib

rf_model = joblib.load("/kaggle/input/knn-imputer/random_forest_regressor_model.joblib")

print("\n--- Training RandomForestRegressor ---")
#rf_model.fit(X_train, y_train) # Use unscaled X_train and y_train directly

print("\nRandomForestRegressor training finished.")


# --- 3. Predict on X_test ---
predictions = rf_model.predict(X_test)

print("\nSample predictions:\n", predictions[:5])
print(f"Number of predictions: {len(predictions)}")


# --- 4. Save Predictions to submission.csv ---

# Create an 'id' column starting from 0
id_column = np.arange(len(predictions))

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'id': id_column,
    'purchaseValue': predictions
})

# Save to CSV
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False) # index=False prevents writing DataFrame index as a column

print(f"\nPredictions saved to {submission_filename}")
print("Submission file head:\n", submission_df.head())